# harden-v0 analysis notebook base

This notebook is for analyzing harden-v0 output data.
Instructions:

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import glob
import os

# define paths
batches_dir = Path("/home/ivgeni/truthserum/shared_harden-v0")

jobs_parquet_path = "anvil_jobs.parquet"
configs_parquet_path = "anvil_configs.parquet"
trials_parquet_path = "anvil_trials.parquet"
trajectories_parquet_path = "anvil_trajectories.parquet"
trial_summaries_parquet_path = "anvil_trial_summaries.parquet"
step_summaries_parquet_path = "anvil_step_summaries.parquet"
tasks_parquet_path = "anvil_tasks.parquet"
iterations_parquet_path = "anvil_iterations.parquet"

# Directory structure:
#   shared_harden-v0/
#     BATCH_DIR/                                ← batch level (e.g. batch_TIMESTAMP/ or
#       batch_summary.json                      ←   ad-hoc names like tb-non-hacked-gemini-3-flash/)
#       batch_config.json
#       TASK_NAME/
#         result.json                           ← task result (status, iterations, hardened_dir, ...)
#         task_config.json
#         hacker_task/  solver_task/  hardened/
#         jobs/
#           JOB_ID/                             ← e.g. solver_precheck_a0__TIMESTAMP__HASH
#             config.json
#             result.json
#             job.log
#             TRIAL_NAME/                       ← e.g. task-name__HASH
#               config.json
#               result.json
#               trial.log
#               agent/
#                 trajectory.json               ← steps, final_metrics
#                 episode-N/                    ← per-episode debug data
#               verifier/  artifacts/
#
# JOB_ID naming: {type}[_iter{N}][_a{N}]__{timestamp}__{hash}
#
# NOTE: trajectory.summary.json files are no longer emitted by current harden-v0
# output, so the summary-related views may be empty.

# Auto-detect batch directories: any immediate subdirectory of `batches_dir`
# that contains at least one `TASK/jobs/` folder is treated as a batch. This
# covers both `batch_TIMESTAMP/` and ad-hoc run dirs (e.g. `tb-non-hacked-*`).
_batch_dirs = sorted(
    p for p in batches_dir.iterdir()
    if p.is_dir() and any(p.glob("*/jobs"))
)
print(f"Detected {len(_batch_dirs)} batch directories: {[p.name for p in _batch_dirs]}")

# Build one glob pattern per batch dir, per relation. DuckDB's read_json_auto
# accepts a list of glob patterns and does file enumeration internally —
# this is far cheaper than passing 1000s of literal paths.
def _globs(suffix):
    return [str(b / suffix) for b in _batch_dirs]

batches_globs = _globs("batch_summary.json")
tasks_globs = _globs("*/result.json")
jobs_globs = _globs("*/jobs/*/result.json")
configs_globs = _globs("*/jobs/*/config.json")
trials_globs = _globs("*/jobs/*/*/result.json")
trajectories_globs = _globs("*/jobs/*/*/agent/trajectory.json")
traj_summaries_globs = _globs("*/jobs/*/*/agent/trajectory.summary.json")

# For sanity-display of file counts (uses Python glob, just for printing):
def _count(patterns):
    n = 0
    for p in patterns:
        n += len(glob.glob(p))
    return n

print(
    f"  batches: {_count(batches_globs)} | tasks: {_count(tasks_globs)} | "
    f"jobs: {_count(jobs_globs)} | configs: {_count(configs_globs)} | "
    f"trials: {_count(trials_globs)} | trajectories: {_count(trajectories_globs)} | "
    f"traj_summaries: {_count(traj_summaries_globs)}"
)

def _sql_globs(patterns):
    """Render a Python list of glob patterns as a DuckDB SQL array literal."""
    return "[" + ", ".join("'" + p.replace("'", "''") + "'" for p in patterns) + "]"

# Connect to an in-memory database
con = duckdb.connect()

# Larger memory limit + temp dir spill — the dataset can be ~10k+ JSON files.
con.execute("SET memory_limit='12GB'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET threads=4")
con.execute("SET temp_directory='/tmp/duckdb_spill'")

def job_id_parse_sql(job_id_expr):
    """SQL fragment: extract job_type, iteration, attempt from a job_id expression."""
    prefix = f"string_split({job_id_expr}, '__')[1]"
    return f"""
        regexp_replace({prefix}, '(_iter[0-9]+)?(_a[0-9]+)?$', '') as job_type,
        try_cast(regexp_extract({prefix}, '_iter([0-9]+)', 1) as INTEGER) as iteration,
        try_cast(regexp_extract({prefix}, '_a([0-9]+)$', 1) as INTEGER) as attempt"""

def convert_to_parquet(name, patterns, parquet_path, build_query):
    """Stream JSON -> Parquet. `patterns` is a list of glob patterns. `build_query(L)`
    returns a SELECT given the SQL array literal."""
    if not patterns or _count(patterns) == 0:
        print(f"No {name} files found.")
        return False
    print(f"Converting {name} files to {parquet_path}...")
    try:
        con.execute(f"COPY ({build_query(_sql_globs(patterns))}) TO '{parquet_path}' (FORMAT PARQUET)")
        con.execute(f"CREATE OR REPLACE VIEW raw_{name} AS SELECT * FROM '{parquet_path}'")
        print(f"Success! Created view 'raw_{name}' from {parquet_path}")
        return True
    except Exception as e:
        print(f"Error converting {name}: {e}")
        return False

# Tasks: BATCH_DIR/TASK_NAME/result.json  ([-3]=batch, [-2]=task)
convert_to_parquet("tasks", tasks_globs, tasks_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-3] as batch_id,
        string_split(filename, '/')[-2] as task_name,
        task_id, status,
        len(iterations) as num_iterations,
        hardened_dir, oracle, pool_enabled, kernelbench_mode
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Iterations (unnested from task results)
convert_to_parquet("iterations", tasks_globs, iterations_parquet_path, lambda L: f"""
    WITH raw AS (
        SELECT
            string_split(filename, '/')[-3] as batch_id,
            string_split(filename, '/')[-2] as task_name,
            task_id,
            unnest(iterations) as iter
        FROM read_json_auto({L}, union_by_name=true, filename=true)
    )
    SELECT
        batch_id, task_name, task_id,
        iter.iteration as iteration,
        iter.hack_reward as hack_reward,
        iter.fix_applied as fix_applied,
        iter.replay_attempted as replay_attempted,
        iter.replay_reward as replay_reward,
        iter.outcome as outcome
    FROM raw
""")

# Jobs: BATCH_DIR/TASK_NAME/jobs/JOB_ID/result.json  ([-5]=batch, [-4]=task, [-2]=job)
convert_to_parquet("jobs", jobs_globs, jobs_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-5] as batch_id,
        string_split(filename, '/')[-4] as task_name,
        string_split(filename, '/')[-2] as job_id,
        {job_id_parse_sql("string_split(filename, '/')[-2]")},
        *
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Configs: BATCH_DIR/TASK_NAME/jobs/JOB_ID/config.json
# environment.path no longer exists in newer configs; falls back to NULL.
convert_to_parquet("configs", configs_globs, configs_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-5] as batch_id,
        string_split(filename, '/')[-4] as task_name,
        string_split(filename, '/')[-2] as job_id,
        {job_id_parse_sql("string_split(filename, '/')[-2]")},
        agents[1].name as agent_name,
        agents[1].model_name as model_name,
        environment.type as env_type,
        try_cast(json_extract(environment, '$.path') as VARCHAR) as env_path
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Trials: BATCH_DIR/TASK_NAME/jobs/JOB_ID/TRIAL_NAME/result.json  ([-6]=batch, [-3]=job)
convert_to_parquet("trials", trials_globs, trials_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-6] as batch_id,
        string_split(filename, '/')[-3] as job_id,
        {job_id_parse_sql("string_split(filename, '/')[-3]")},
        *
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Trajectories: BATCH_DIR/TASK_NAME/jobs/JOB_ID/TRIAL_NAME/agent/trajectory.json
# Step schema: {step_id, timestamp, source, message, [model_name, reasoning_content, tool_calls, observation, metrics]}
# Only agent-source steps carry .metrics (prompt_tokens, completion_tokens, cost_usd).
convert_to_parquet("trajectories", trajectories_globs, trajectories_parquet_path, lambda L: f"""
    SELECT
        string_split(filename, '/')[-7] as batch_id,
        string_split(filename, '/')[-6] as task_name,
        string_split(filename, '/')[-4] as job_id,
        {job_id_parse_sql("string_split(filename, '/')[-4]")},
        string_split(filename, '/')[-3] as trial_name,
        unnest(steps) as step,
        * EXCLUDE (steps)
    FROM read_json_auto({L}, union_by_name=true, filename=true)
""")

# Trajectory Summaries (legacy; usually empty for current harden-v0 output)
if _count(traj_summaries_globs) > 0:
    L = _sql_globs(traj_summaries_globs)
    print(f"Converting trajectory summary files...")
    try:
        con.execute(f"""
            COPY (
                SELECT
                    string_split(filename, '/')[-7] as batch_id,
                    string_split(filename, '/')[-6] as task_name,
                    string_split(filename, '/')[-4] as job_id,
                    {job_id_parse_sql("string_split(filename, '/')[-4]")},
                    string_split(filename, '/')[-3] as trial_name,
                    summary.summary as trial_summary,
                    summary.metrics as metrics,
                    summary.questionnaire as questionnaire
                FROM read_json_auto({L}, union_by_name=true, filename=true)
            ) TO '{trial_summaries_parquet_path}' (FORMAT PARQUET)
        """)
        con.execute(f"CREATE OR REPLACE VIEW raw_trial_summaries AS SELECT * FROM '{trial_summaries_parquet_path}'")
        print("Created raw_trial_summaries view")
    except Exception as e:
        print(f"Error processing trial summaries: {e}")
    try:
        con.execute(f"""
            COPY (
                WITH raw_steps AS (
                    SELECT
                        string_split(filename, '/')[-7] as batch_id,
                        string_split(filename, '/')[-6] as task_name,
                        string_split(filename, '/')[-4] as job_id,
                        {job_id_parse_sql("string_split(filename, '/')[-4]")},
                        string_split(filename, '/')[-3] as trial_name,
                        unnest(steps) as step_info
                    FROM read_json_auto({L}, union_by_name=true, filename=true)
                )
                SELECT
                    batch_id, task_name, job_id, job_type, iteration, attempt, trial_name,
                    step_info.step_id as step_id,
                    step_info.source as source,
                    step_info.summary as summary
                FROM raw_steps
            ) TO '{step_summaries_parquet_path}' (FORMAT PARQUET)
        """)
        con.execute(f"CREATE OR REPLACE VIEW flat_step_summaries AS SELECT * FROM '{step_summaries_parquet_path}'")
        print("Created flat_step_summaries view")
    except Exception as e:
        print(f"Error processing step summaries: {e}")
else:
    print("No trajectory summary files found.")

# Print summary stats
print("\n" + "="*70)
print("PARQUET FILE STATISTICS")
print("="*70)

parquet_files = [
    ("Tasks", tasks_parquet_path, "raw_tasks"),
    ("Iterations", iterations_parquet_path, "raw_iterations"),
    ("Jobs", jobs_parquet_path, "raw_jobs"),
    ("Configs", configs_parquet_path, "raw_configs"),
    ("Trials", trials_parquet_path, "raw_trials"),
    ("Trajectories", trajectories_parquet_path, "raw_trajectories"),
    ("Trial Summaries", trial_summaries_parquet_path, "raw_trial_summaries"),
    ("Step Summaries", step_summaries_parquet_path, "flat_step_summaries"),
    ("Pool Commits", pool_commits_parquet_path, "raw_pool_commits"),
]

for name, file_path, view_name in parquet_files:
    try:
        if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
            row_count = con.execute(f"SELECT COUNT(*) FROM '{file_path}'").fetchall()[0][0]
            file_size_mb = os.path.getsize(file_path) / (1024*1024)
            file_size_gb = file_size_mb / 1024
            col_count = len(con.execute(f"SELECT * FROM '{file_path}' LIMIT 0").description)
            size_str = f"{file_size_gb:.2f} GB" if file_size_gb >= 1 else f"{file_size_mb:.2f} MB"
            print(f"\n{name:20} | Records: {row_count:>10,} | Columns: {col_count:>3} | Size: {size_str:>10}")
        else:
            print(f"\n{name:20} | (no data)")
    except Exception as e:
        print(f"\n{name:20} | Error: {e}")

print("\n" + "="*70)

Detected 2 batch directories: ['batch_20260421_034837', 'tb-non-hacked-gemini-3-flash']
  batches: 2 | tasks: 198 | jobs: 3057 | configs: 3059 | trials: 3057 | trajectories: 2922 | traj_summaries: 0
Converting tasks files to anvil_tasks.parquet...
Success! Created view 'raw_tasks' from anvil_tasks.parquet
Converting iterations files to anvil_iterations.parquet...
Success! Created view 'raw_iterations' from anvil_iterations.parquet
Converting jobs files to anvil_jobs.parquet...
Success! Created view 'raw_jobs' from anvil_jobs.parquet
Converting configs files to anvil_configs.parquet...
Success! Created view 'raw_configs' from anvil_configs.parquet
Converting trials files to anvil_trials.parquet...
Success! Created view 'raw_trials' from anvil_trials.parquet
Converting trajectories files to anvil_trajectories.parquet...
Success! Created view 'raw_trajectories' from anvil_trajectories.parquet
No trajectory summary files found.

PARQUET FILE STATISTICS

Tasks                | Records:     

In [13]:

# =============================================================================
# Pool.git commit extraction
# =============================================================================
# Each batch directory may contain a bare `pool.git` repo at:
#   shared_harden-v0/<BATCH_DIR>/pool.git
# Commits look like:
#   [bootstrap] from <source-path>
#   [<task-name> iter-<N>] <tag>: <description>
# We extract every commit on every ref and parse out the structured prefix so
# rows can be joined to `tasks` (by batch_id + task_name) or `iterations`
# (by batch_id + task_name + iteration).
import subprocess
import re

pool_commits_parquet_path = "anvil_pool_commits.parquet"

# git log uses unit-separator (US, \x1f) between fields, record-separator (RS,
# \x1e) between commits — neither appears in normal commit text.
_GIT_LOG_FORMAT = (
    "%H%x1f%P%x1f%T%x1f%aI%x1f%cI%x1f%an%x1f%ae%x1f%cn%x1f%ce%x1f%D%x1f%s%x1f%b%x1e"
)
_GIT_FIELDS = [
    "commit_sha", "parents_raw", "tree_sha",
    "author_when", "committer_when",
    "author_name", "author_email",
    "committer_name", "committer_email",
    "refs_raw", "subject", "body",
]
# Subject patterns:
#   [bootstrap] from <source-path>
#   [<task-name> iter-<N>] <tag>: <description>
_BOOTSTRAP_RE = re.compile(r"^\[bootstrap\]\s+from\s+(?P<source_path>.+)$")
_ITER_RE = re.compile(
    r"^\[(?P<task_name>[^\]\s]+)\s+iter-(?P<iteration>\d+)\]\s*(?:(?P<tag>[^:]+?):\s*)?(?P<description>.*)$"
)


def _parse_subject(subject):
    m = _BOOTSTRAP_RE.match(subject or "")
    if m:
        return {
            "message_kind": "bootstrap",
            "task_name": None,
            "iteration": None,
            "tag": None,
            "description": None,
            "source_path": m.group("source_path").strip(),
        }
    m = _ITER_RE.match(subject or "")
    if m:
        return {
            "message_kind": "iter",
            "task_name": m.group("task_name"),
            "iteration": int(m.group("iteration")),
            "tag": (m.group("tag") or "").strip() or None,
            "description": (m.group("description") or "").strip() or None,
            "source_path": None,
        }
    return {
        "message_kind": "other",
        "task_name": None,
        "iteration": None,
        "tag": None,
        "description": None,
        "source_path": None,
    }


def _extract_pool_commits(batch_dir):
    """Return a list of commit dicts for the given batch dir's pool.git, or []."""
    pool_git = batch_dir / "pool.git"
    if not pool_git.is_dir():
        return []
    try:
        proc = subprocess.run(
            ["git", "--git-dir", str(pool_git), "log", "--all",
             f"--format={_GIT_LOG_FORMAT}", "--decorate=full"],
            capture_output=True, text=True, check=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"  ⚠ git log failed in {pool_git}: {e.stderr.strip()}")
        return []

    rows = []
    for raw in proc.stdout.split("\x1e"):
        raw = raw.strip("\n")
        if not raw:
            continue
        fields = raw.split("\x1f")
        if len(fields) < len(_GIT_FIELDS):
            # pad missing trailing fields (e.g. empty body)
            fields = fields + [""] * (len(_GIT_FIELDS) - len(fields))
        rec = dict(zip(_GIT_FIELDS, fields))
        rec["batch_id"] = batch_dir.name
        parents = rec.pop("parents_raw").strip()
        rec["parent_shas"] = parents.split() if parents else []
        refs = rec.pop("refs_raw").strip()
        rec["refs"] = [r.strip() for r in refs.split(",") if r.strip()] if refs else []
        rec["body"] = rec["body"].strip() or None
        rec.update(_parse_subject(rec["subject"]))
        rows.append(rec)
    return rows


pool_commit_rows = []
batches_with_pool = 0
for _b in _batch_dirs:
    rows = _extract_pool_commits(_b)
    if rows:
        batches_with_pool += 1
        pool_commit_rows.extend(rows)

print(
    f"Pool.git: scanned {len(_batch_dirs)} batches, "
    f"{batches_with_pool} have pool.git, extracted {len(pool_commit_rows)} commits"
)

if pool_commit_rows:
    pool_df = pd.DataFrame(pool_commit_rows)
    # Cast timestamp columns
    pool_df["author_when"] = pd.to_datetime(pool_df["author_when"], utc=True, errors="coerce")
    pool_df["committer_when"] = pd.to_datetime(pool_df["committer_when"], utc=True, errors="coerce")
    # Reorder for readability
    pool_df = pool_df[[
        "batch_id", "commit_sha", "parent_shas", "tree_sha",
        "author_when", "committer_when",
        "author_name", "author_email", "committer_name", "committer_email",
        "refs", "message_kind", "task_name", "iteration", "tag",
        "description", "subject", "body", "source_path",
    ]]
    con.register("_pool_df", pool_df)
    con.execute(
        f"COPY (SELECT * FROM _pool_df) TO '{pool_commits_parquet_path}' (FORMAT PARQUET)"
    )
    con.unregister("_pool_df")
    con.execute(
        f"CREATE OR REPLACE VIEW raw_pool_commits AS SELECT * FROM '{pool_commits_parquet_path}'"
    )
    print(f"✓ Wrote {pool_commits_parquet_path} ({len(pool_df)} rows) and view raw_pool_commits")
else:
    # Drop any stale parquet so downstream stats show "(no data)"
    if os.path.exists(pool_commits_parquet_path):
        os.remove(pool_commits_parquet_path)
    print("No pool.git commits to write.")


Pool.git: scanned 2 batches, 1 have pool.git, extracted 294 commits
✓ Wrote anvil_pool_commits.parquet (294 rows) and view raw_pool_commits


In [14]:
# GLOBAL JOB FILTER CONFIGURATION
# =============================================================================
from IPython.display import display

## FILTER CONFIGURATION:
## Add list of batch IDs to filter by, and re-run ALL cells from here to end
FILTERED_BATCH_IDS = None
FILTERED_BATCH_IDS = ['tb-non-hacked-gemini-3-flash']

def create_filtered_views(batch_ids):
    """Create filtered views for all tables based on batch_ids filter."""
    if batch_ids:
        ids_str = ", ".join([f"'{bid}'" for bid in batch_ids])
        batch_filter = f"WHERE batch_id IN ({ids_str})"
    else:
        batch_filter = ""
    
    views = [
        ("tasks", tasks_parquet_path),
        ("iterations", iterations_parquet_path),
        ("jobs", jobs_parquet_path),
        ("configs", configs_parquet_path),
        ("trials", trials_parquet_path),
        ("trajectories", trajectories_parquet_path),
        ("trial_summaries", trial_summaries_parquet_path),
        ("step_summaries", step_summaries_parquet_path),
        ("pool_commits", pool_commits_parquet_path),
    ]
    
    for name, parquet_path in views:
        if not (os.path.exists(parquet_path) and os.path.getsize(parquet_path) > 0):
            continue
        try:
            con.execute(f"""
                CREATE OR REPLACE VIEW {name} AS 
                SELECT * FROM '{parquet_path}' {batch_filter}
            """)
        except Exception as e:
            print(f"Warning: Could not create view '{name}': {e}")
    
    filter_desc = f"{len(batch_ids)} batch(es) selected" if batch_ids else "ALL BATCHES (no filter)"
    print(f"✓ Filters updated: {filter_desc}")

create_filtered_views(FILTERED_BATCH_IDS)
# Initialize views with default selection

✓ Filters updated: 1 batch(es) selected


In [15]:
# Display schema for every view
view_names = ["tasks", "iterations", "jobs", "configs", "trials", "trajectories", "trial_summaries", "step_summaries", "pool_commits"]

for view_name in view_names:
    try:
        print(f"\n{'='*60}")
        print(f"Schema: {view_name}")
        print(f"{'='*60}")
        display(con.execute(f"DESCRIBE {view_name}").df())
    except Exception as e:
        print(f"  ⚠ View '{view_name}' not available: {e}")


Schema: tasks


,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,task_name,VARCHAR,YES,None,None,None
2,task_id,VARCHAR,YES,None,None,None
3,status,VARCHAR,YES,None,None,None
4,num_iterations,BIGINT,YES,None,None,None
5,hardened_dir,VARCHAR,YES,None,None,None
6,oracle,BOOLEAN,YES,None,None,None
7,pool_enabled,BOOLEAN,YES,None,None,None
8,kernelbench_mode,BOOLEAN,YES,None,None,None



Schema: iterations


,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,task_name,VARCHAR,YES,None,None,None
2,task_id,VARCHAR,YES,None,None,None
3,iteration,BIGINT,YES,None,None,None
4,hack_reward,DOUBLE,YES,None,None,None
5,fix_applied,BOOLEAN,YES,None,None,None
6,replay_attempted,BOOLEAN,YES,None,None,None
7,replay_reward,JSON,YES,None,None,None
8,outcome,VARCHAR,YES,None,None,None



Schema: jobs


...

,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,job_id,VARCHAR,YES,None,None,None
2,job_type,VARCHAR,YES,None,None,None
3,iteration,INTEGER,YES,None,None,None
4,attempt,INTEGER,YES,None,None,None
5,id,UUID,YES,None,None,None
6,task_name,VARCHAR,YES,None,None,None
7,trial_name,VARCHAR,YES,None,None,None
8,trial_uri,VARCHAR,YES,None,None,None
9,task_id,STRUCT(path VARCHAR),YES,None,None,None



Schema: trajectories


,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,task_name,VARCHAR,YES,None,None,None
2,job_id,VARCHAR,YES,None,None,None
3,job_type,VARCHAR,YES,None,None,None
4,iteration,INTEGER,YES,None,None,None
5,attempt,INTEGER,YES,None,None,None
6,trial_name,VARCHAR,YES,None,None,None
7,step,"STRUCT(step_id BIGINT, ""timestamp"" VARCHAR, ""s...",YES,None,None,None
8,schema_version,VARCHAR,YES,None,None,None
9,session_id,UUID,YES,None,None,None



Schema: trial_summaries
  ⚠ View 'trial_summaries' not available: Catalog Error: Table with name trial_summaries does not exist!
Did you mean "trials"?

LINE 1: DESCRIBE trial_summaries
                 ^

Schema: step_summaries
  ⚠ View 'step_summaries' not available: Catalog Error: Table with name step_summaries does not exist!
Did you mean "sqlite_temp_master"?

LINE 1: DESCRIBE step_summaries
                 ^

Schema: pool_commits


,column_name,column_type,null,key,default,extra
0,batch_id,VARCHAR,YES,None,None,None
1,commit_sha,VARCHAR,YES,None,None,None
2,parent_shas,VARCHAR[],YES,None,None,None
3,tree_sha,VARCHAR,YES,None,None,None
4,author_when,TIMESTAMP WITH TIME ZONE,YES,None,None,None
5,committer_when,TIMESTAMP WITH TIME ZONE,YES,None,None,None
6,author_name,VARCHAR,YES,None,None,None
7,author_email,VARCHAR,YES,None,None,None
8,committer_name,VARCHAR,YES,None,None,None
9,committer_email,VARCHAR,YES,None,None,None


In [9]:
# Tasks Overview
print("=== All Tasks (sorted by status) ===")
display(con.execute("""
    SELECT * FROM tasks ORDER BY status, task_name
""").df())

print("\n=== Task Status Aggregation Per Batch ===")
display(con.execute("""
    SELECT 
        batch_id,
        status,
        COUNT(*) as count
    FROM tasks
    GROUP BY batch_id, status
    ORDER BY batch_id, count DESC
""").df())

=== All Tasks (sorted by status) ===


,batch_id,task_name,task_id,status,num_iterations,hardened_dir,oracle,pool_enabled,kernelbench_mode
0,tb-non-hacked-gemini-3-flash,acl-permissions-inheritance,acl-permissions-inheritance,max_iterations,10,outputs/tb-non-hacked-gemini-3-flash/acl-permi...,False,True,False
1,tb-non-hacked-gemini-3-flash,aimo-airline-departures,aimo-airline-departures,max_iterations,10,outputs/tb-non-hacked-gemini-3-flash/aimo-airl...,False,True,False
2,tb-non-hacked-gemini-3-flash,analyze-access-logs,analyze-access-logs,max_iterations,10,outputs/tb-non-hacked-gemini-3-flash/analyze-a...,False,True,False
3,tb-non-hacked-gemini-3-flash,assign-seats,assign-seats,max_iterations,10,outputs/tb-non-hacked-gemini-3-flash/assign-se...,False,True,False
4,tb-non-hacked-gemini-3-flash,attention-mil,attention-mil,max_iterations,10,outputs/tb-non-hacked-gemini-3-flash/attention...,False,True,False
...,...,...,...,...,...,...,...,...,...
191,tb-non-hacked-gemini-3-flash,stable-parallel-kmeans,stable-parallel-kmeans,solver_failed_precheck,0,None,False,True,False
192,tb-non-hacked-gemini-3-flash,torch-pipeline-parallelism,torch-pipeline-parallelism,solver_failed_precheck,0,None,False,True,False
193,tb-non-hacked-gemini-3-flash,unprivileged-headless-pyrender,unprivileged-headless-pyrender,solver_failed_precheck,0,None,False,True,False
194,tb-non-hacked-gemini-3-flash,word2vec-from-scratch,word2vec-from-scratch,solver_failed_precheck,0,None,False,True,False



=== Task Status Aggregation Per Batch ===


,batch_id,status,count
0,tb-non-hacked-gemini-3-flash,max_iterations,127
1,tb-non-hacked-gemini-3-flash,solver_failed_precheck,54
2,tb-non-hacked-gemini-3-flash,robust,15


In [10]:
# Token Usage & Cost Aggregation
# Uses agent_result from trial result.json (not trajectory summaries)
import math

def fmt_tokens(n):
    """Format token count as human-readable string (e.g. 1.4M, 523.1K)"""
    if n is None or (isinstance(n, float) and math.isnan(n)):
        return "—"
    n = float(n)
    if abs(n) >= 1_000_000_000:
        return f"{n / 1_000_000_000:.1f}B"
    if abs(n) >= 1_000_000:
        return f"{n / 1_000_000:.1f}M"
    if abs(n) >= 1_000:
        return f"{n / 1_000:.1f}K"
    return str(int(n))

def fmt_cost(c):
    if c is None or (isinstance(c, float) and math.isnan(c)):
        return "—"
    return f"${c:.4f}"

def fmt_duration(seconds):
    """Format seconds into human-readable duration (e.g. 12h 49m)"""
    if seconds is None or (isinstance(seconds, float) and math.isnan(seconds)):
        return "—"
    seconds = int(seconds)
    if seconds < 60:
        return f"{seconds}s"
    minutes = seconds // 60
    secs = seconds % 60
    if minutes < 60:
        return f"{minutes}m {secs}s"
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours}h {mins}m"

token_cols = ['total_input_tokens', 'total_output_tokens', 'total_cached_tokens']
avg_token_cols = ['avg_input_tokens', 'avg_output_tokens']
median_token_cols = ['median_input_tokens', 'median_output_tokens']


print("\n=== Token Usage Per Task ===")
query_tokens_per_task = """
    WITH last_steps AS (
        -- Get the last step per trial (max step_id) and its context length
        SELECT
            traj.batch_id,
            traj.job_id,
            traj.trial_name,
            traj.step.metrics.prompt_tokens + traj.step.metrics.completion_tokens as trajectory_length
        FROM trajectories traj
        WHERE traj.step.step_id = (
            SELECT MAX(t2.step.step_id)
            FROM trajectories t2
            WHERE t2.batch_id = traj.batch_id AND t2.job_id = traj.job_id AND t2.trial_name = traj.trial_name
        )
    ),
    trial_step_counts AS (
        -- Count total steps per trial
        SELECT
            traj.batch_id,
            traj.job_id,
            traj.trial_name,
            COUNT(*) as num_steps
        FROM trajectories traj
        GROUP BY traj.batch_id, traj.job_id, traj.trial_name
    )
    SELECT 
        t.task_name,
        COUNT(*) as num_trials,
        COUNT(*) FILTER (WHERE t.verifier_result.rewards.reward > 0) as num_rewarded,
        COUNT(*) FILTER (WHERE t.exception_info IS NOT NULL) as num_errors,
        SUM(t.agent_result.n_input_tokens) as total_input_tokens,
        SUM(t.agent_result.n_output_tokens) as total_output_tokens,
        SUM(t.agent_result.n_cache_tokens) as total_cached_tokens,
        SUM(t.agent_result.cost_usd) as total_cost,
        CAST(AVG(t.agent_result.n_input_tokens) AS BIGINT) as avg_input_tokens,
        CAST(AVG(t.agent_result.n_output_tokens) AS BIGINT) as avg_output_tokens,
        MAX(ls.trajectory_length) as max_trajectory_length,
        MAX(tsc.num_steps) as max_steps
    FROM trials t
    LEFT JOIN last_steps ls ON t.batch_id = ls.batch_id AND t.job_id = ls.job_id AND t.trial_name = ls.trial_name
    LEFT JOIN trial_step_counts tsc ON t.batch_id = tsc.batch_id AND t.job_id = tsc.job_id AND t.trial_name = tsc.trial_name
    GROUP BY t.task_name
    ORDER BY SUM(t.agent_result.n_input_tokens) DESC
"""
try:
    df_task = con.execute(query_tokens_per_task).df()
    for col in token_cols + avg_token_cols + ['max_trajectory_length']:
        if col in df_task.columns:
            df_task[col] = df_task[col].apply(fmt_tokens)
    if 'total_cost' in df_task.columns:
        df_task['total_cost'] = df_task['total_cost'].apply(fmt_cost)
    display(df_task)
except Exception as e:
    print(f"Error: {e}")


=== Token Usage Per Task ===


,task_name,num_trials,num_rewarded,num_errors,total_input_tokens,total_output_tokens,total_cached_tokens,total_cost,avg_input_tokens,avg_output_tokens,max_trajectory_length,max_steps
0,vul-flink,14,1,0,22.9M,322.0K,19.5M,$3.6523,1.6M,23.0K,181.9K,61
1,png-generation,19,4,2,22.7M,218.7K,20.2M,$2.8998,1.2M,11.5K,339.1K,56
2,fix-ocaml-gc,16,5,5,22.3M,203.6K,18.0M,$3.6702,1.4M,12.7K,734.3K,61
3,port-compressor,4,0,0,22.0M,129.0K,20.0M,$2.4213,5.5M,32.3K,206.7K,63
4,build-linux-kernel-qemu,18,5,1,21.0M,242.7K,17.2M,$3.4904,1.2M,13.5K,180.4K,61
...,...,...,...,...,...,...,...,...,...,...,...,...
191,shell-deobfuscation,4,0,4,—,—,—,—,—,—,—,<NA>
192,privilege-escalation,4,0,4,—,—,—,—,—,—,—,<NA>
193,recover-accuracy-log,4,0,4,—,—,—,—,—,—,—,<NA>
194,unprivileged-headless-pyrender,4,0,4,—,—,—,—,—,—,—,<NA>
